# Hierarchischer Ansatz: PLS-DA

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** PLS-DA je Stufe im hierarchischen Ansatz testen (#100).

In [ ]:
%load_ext autoreload
%autoreload 2

import ast

import pandas as pd
from mlflow import MlflowClient

from awp2.config import MLFLOW_TRACKING_URI, MODEL_DOCS_FIGURES_DIR
from awp2.data import load_split
from awp2.evaluation import compare_predictions, error_breakdown
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import HierarchicalClassifier, make_hierarchical_svm, oracle_crop_metrics
from awp2.models.plsda import PLSDAClassifier
from awp2.plots import plot_error_breakdown, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

PLS-DA ist ein Standard der Spektroskopie: PLS verdichtet die stark korrelierten Bänder überwacht auf wenige Komponenten, die mit den Klassen kovariieren; eine LDA mit gleichen Priors klassifiziert darauf (`PLSDAClassifier`). Frage: Hilft die überwachte Verdichtung den linearen Modellen (LDA 0.657, logistische Regression 0.747) näher an die RBF-SVM?

Gleicher Ablauf wie bei der Referenz `hierarchical_svm` (#87): CV-Suche mit `tune()` auf dem
Trainingsteil, dann genau ein `run()`. Verglichen wird per CV mit der SVM-Referenz, danach
Fehlerzerlegung, Oracle-Kultur und McNemar gegen die SVM auf der Validierung.

In [ ]:
SVM_SEARCH_RUN_ID = "2b851b8899c64828995dee02d90a5299"
svm_search = MlflowClient(MLFLOW_TRACKING_URI).get_run(SVM_SEARCH_RUN_ID).data
svm_cv_score = svm_search.metrics["cv_bacc_combined_mean"]
svm_params = {
    name.removeprefix("best."): ast.literal_eval(value)
    for name, value in svm_search.params.items()
    if name.startswith("best.")
}
svm_preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
TOP_CANDIDATES = 15

model = HierarchicalClassifier(PLSDAClassifier(), PLSDAClassifier())

components = [10, 20, 40, 80, 120]
param_grid = {
    "crop_estimator__n_components": components,
    "stage_estimator__n_components": components,
}

## CV-Suche

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="hierarchical_plsda_search",
        approach="hierarchical",
        description="PLS-DA je Stufe im hierarchischen Ansatz (#100).",
        preprocessing=preprocessing,
        balance_samples=False,
        param_grid=param_grid,
    ),
)
tuned.best_params, round(tuned.best_score, 4), round(tuned.best_score - svm_cv_score, 4)

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="crop_estimator__n_components"
)
tuning_ax.axvline(svm_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.set_title("Hierarchisch PLS-DA: CV-Kandidaten (gestrichelt: SVM-Referenz)")
save_doc_figure(tuning_ax.figure, "hierarchical_plsda_tuning", MODEL_DOCS_FIGURES_DIR)

## Einmalige Bewertung

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="hierarchical_plsda",
        approach="hierarchical",
        description="Bester PLS-DA aus hierarchical_plsda_search, einmal bewertet.",
        preprocessing=preprocessing,
        balance_samples=False,
        tuning_run=tuned.run_id,
    ),
)
result.metrics.model_dump(), result.run_id

## Vergleich mit der SVM-Referenz

Die SVM-Referenz wird mit ihren protokollierten Einstellungen nachtrainiert (`track=False`,
kein neuer Lauf), um die Vorhersagen Zeile für Zeile zu vergleichen.

In [ ]:
svm_result = run(
    make_hierarchical_svm().set_params(**svm_params),
    RunConfig(
        name="hierarchical_svm_refit",
        approach="hierarchical",
        preprocessing=svm_preprocessing,
        track=False,
        log_model=False,
    ),
)
breakdowns = {
    "RBF-SVM (Referenz)": error_breakdown(split.y_val, svm_result.y_pred),
    "PLS-DA": error_breakdown(split.y_val, result.y_pred),
}
oracle = {
    "RBF-SVM (Referenz)": oracle_crop_metrics(svm_result.pipeline, split.X_val, split.y_val),
    "PLS-DA": oracle_crop_metrics(result.pipeline, split.X_val, split.y_val),
}
comparison = pd.DataFrame(
    {
        label: {
            "BAcc kombiniert": r.metrics.bacc_combined,
            "Oracle-Kultur kombiniert": oracle[label].bacc_combined,
            **breakdowns[label].model_dump(),
        }
        for label, r in {"RBF-SVM (Referenz)": svm_result, "PLS-DA": result}.items()
    }
)
comparison.round(3), compare_predictions(split.y_val, svm_result.y_pred, result.y_pred)

In [ ]:
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung: PLS-DA vs. RBF-SVM")
save_doc_figure(breakdown_ax.figure, "hierarchical_plsda_breakdown", MODEL_DOCS_FIGURES_DIR)

## Befund

- CV 0.678, Validierung 0.686 – kaum besser als LDA (0.657 / 0.676), weit unter der SVM.
- Die Kulturstufe wählt 120 Komponenten (Rasterrand), also fast keine Verdichtung; das Stadium
  kommt mit 20 aus. Überwachte Verdichtung bringt den linearen Modellen keinen Durchbruch.
- 20 % falsche Kultur, mit wahrer Kultur 0.898.